# Lesson 4: Persistence and Streaming

In [ ]:
from dotenv import load_dotenv

_ = load_dotenv()

In [1]:
from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from langchain_community.tools import DuckDuckGoSearchRun

In [2]:
#tool = TavilySearchResults(max_results=2)
tool = DuckDuckGoSearchRun()

In [3]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [4]:
from langgraph.checkpoint.sqlite import SqliteSaver



In [5]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [6]:
prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""
#model = ChatOpenAI(model="gpt-4o")
base_url = "http://127.0.0.1:1234/v1" 
model_name ="google/gemma-3-4b"

model = ChatOpenAI(model=model_name, temperature=0, max_tokens=None, timeout=None, max_retries=2,api_key="llm-studio", base_url=base_url)


In [7]:
with SqliteSaver.from_conn_string(":memory:") as checkpointer:
    
    agent = Agent(model, [tool], checkpointer, system=prompt)

    messages = [HumanMessage(content="What is the weather in SF?")]
    thread = {"configurable": {"thread_id": "1"}}

    # Streaming de eventos
    for event in agent.graph.stream({"messages": messages}, thread):
        for value in event.values():
            print(value['messages'])


    messages = [HumanMessage(content="What about in la?")]
    thread = {"configurable": {"thread_id": "1"}}
    for event in  agent.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)
    messages = [HumanMessage(content="Which one is warmer?")]
    thread = {"configurable": {"thread_id": "1"}}
    for event in  agent.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v) 
    messages = [HumanMessage(content="Which one is warmer?")]
    thread = {"configurable": {"thread_id": "2"}}
    for event in  agent.graph.stream({"messages": messages}, thread):
        for v in event.values():
            print(v)

        


[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '743142839', 'function': {'arguments': '{"query":"weather in San Francisco"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 36, 'prompt_tokens': 498, 'total_tokens': 534, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-h8qn3k36ehg82yh6wn6za', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--441ace82-03e9-4573-9111-036fbac41fd8-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'weather in San Francisco'}, 'id': '743142839', 'type': 'tool_call'}], usage_metadata={'input_tokens': 498, 'output_tokens': 36, 'total_tokens': 534, 'input_token_details': {}, 'output_token_details': {}})]
Calling: {'name': 'duckduckgo_search', 'args': {'query': 'weather in San Francisco'}, 'id': '743142839', 'type

## Streaming tokens

In [ ]:
from langgraph.checkpoint.aiosqlite import AsyncSqliteSaver


memory = AsyncSqliteSaver.from_conn_string(":memory:")
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [ ]:
messages = [HumanMessage(content="What is the weather in SF?")]
thread = {"configurable": {"thread_id": "4"}}
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content:
            # Empty content in the context of OpenAI means
            # that the model is asking for a tool to be invoked.
            # So we only print non-empty content
            print(content, end="|")

In [12]:

from langgraph.checkpoint.sqlite.aio import AsyncSqliteSaver
with AsyncSqliteSaver.from_conn_string(":memory:") as checkpointer:
    abot = Agent(model, [tool], system=prompt, checkpointer=checkpointer)
    messages = [HumanMessage(content="What is the weather in SF?")]
    thread = {"configurable": {"thread_id": "4"}}
    async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
        kind = event["event"]
        if kind == "on_chat_model_stream":
            content = event["data"]["chunk"].content
            if content:
                # Empty content in the context of OpenAI means
                # that the model is asking for a tool to be invoked.
                # So we only print non-empty content
                print(content, end="|")
    
    

AttributeError: __enter__

Calling: {'name': 'duckduckgo_search', 'args': {'query': 'weather in San Francisco'}, 'id': '176516052', 'type': 'tool_call'}
Back to the model!


In [8]:
# another version of synchronour code
from contextlib import ExitStack

stack = ExitStack()
memory = stack.enter_context(SqliteSaver.from_conn_string(":memory:"))

abot = Agent(model, [tool], system=prompt, checkpointer=memory)
messages = [HumanMessage(content="What is the weather in Issaquah?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

messages = [HumanMessage(content="What about in la?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

messages = [HumanMessage(content="Which one is warmer?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

# Exit all contexts
stack.close()

[AIMessage(content='', additional_kwargs={'tool_calls': [{'id': '103744733', 'function': {'arguments': '{"query":"weather in Issaquah"}', 'name': 'duckduckgo_search'}, 'type': 'function'}], 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 37, 'prompt_tokens': 500, 'total_tokens': 537, 'completion_tokens_details': None, 'prompt_tokens_details': None}, 'model_name': 'google/gemma-3-4b', 'system_fingerprint': 'google/gemma-3-4b', 'id': 'chatcmpl-ltx5nd9kmjqwsg1hdol26', 'service_tier': None, 'finish_reason': 'tool_calls', 'logprobs': None}, id='run--ebb2baf0-7f34-473a-bc7f-f88b3bafa152-0', tool_calls=[{'name': 'duckduckgo_search', 'args': {'query': 'weather in Issaquah'}, 'id': '103744733', 'type': 'tool_call'}], usage_metadata={'input_tokens': 500, 'output_tokens': 37, 'total_tokens': 537, 'input_token_details': {}, 'output_token_details': {}})]
Calling: {'name': 'duckduckgo_search', 'args': {'query': 'weather in Issaquah'}, 'id': '103744733', 'type': 'tool_call'}